# 🕳️ 10.14 Handling Nulls & Missing Data

Welcome back! In our last lesson (**10.13 Complex Data Types & Nested Data**), we learned how to unpack and flatten highly nested JSON structures.

But there is a silent killer lurking in almost every real-world dataset: **Missing Data**. 

In the perfect world of tutorials, every cell in a table has a value. In the real world, sensors break, users skip optional form fields, and network requests drop. This results in empty cells, represented in databases and Spark as **Null**.

If you try to perform math on a Null value (e.g., `100 + Null`), the result isn't 100. It is `Null`. If left unchecked, Nulls will silently destroy your analytics and crash Machine Learning models. In this lesson, we will learn how Data Engineers safely detect, drop, and fill these missing values.

### 🎯 Learning Objectives
* Understand the danger of `Null` values in Big Data pipelines.
* Learn how to aggressively remove missing data using `dropna()`.
* Learn how to intelligently patch missing data using `fillna()`.
* Use `replace()` to fix "fake" nulls (like the string "N/A").
* Master **Null-Safe Operations** when writing conditional logic.

In [ ]:
# 0. Setup: Let's start our SparkSession and create a DataFrame full of holes!
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Handling_Nulls").master("local[*]").getOrCreate()

# Python uses 'None' to represent a missing value. 
# Spark will automatically convert 'None' into its native 'null'.
messy_data = [
    (1, "Alice", 85000.0, "Sales"),
    (2, "Bob", None, "Engineering"),         # Bob is missing a salary
    (3, "Charlie", 60000.0, None),           # Charlie is missing a department
    (4, None, None, None),                   # Row 4 is completely blank except for the ID
    (5, "Eve", 95000.0, "N/A")               # Eve has a 'fake' null (a string that says N/A)
]
columns = ["id", "name", "salary", "department"]

df_messy = spark.createDataFrame(messy_data, columns)
print("Raw DataFrame with Missing Data:")
df_messy.show()

## 1. Dropping Nulls: `dropna()`

The most aggressive way to handle missing data is simply to throw it away. We use the `.dropna()` method (which is an alias for `.na.drop()`).

You can configure exactly how aggressive you want Spark to be:
* `how="any"` (Default): Drops the entire row if *even one single column* contains a Null.
* `how="all"`: Only drops the row if *every single column* is Null.
* `subset=["col1", "col2"]`: Only checks specific columns for Nulls. If those specific columns are Null, the row is dropped.

<img src="../assets/Module_10/10_14_01.png" alt="A visual of a swiss cheese board with holes representing nulls. One arrow shows throwing the entire slice of cheese away into a trash can (dropna), while another arrow shows patching the holes with perfectly cut pieces of new cheese (fillna)." width="75%">

In [ ]:
print("--- Strategy 1: Drop ANY row with ANY missing data ---")
df_drop_any = df_messy.dropna(how="any")
df_drop_any.show()
# Notice that ONLY Alice survived! Everyone else had at least one missing piece of data.

print("--- Strategy 2: Drop only if specific columns are null ---")
# In the real world, maybe we don't care if 'department' is missing,
# but a missing 'name' is unacceptable.
df_drop_subset = df_messy.dropna(subset=["name"])
df_drop_subset.show()
# Notice Row 4 was dropped (no name), but Bob and Charlie survived.

## 2. Filling Nulls: `fillna()`

Throwing data away is often a bad idea. If you drop every row with a missing field, you might lose 50% of your dataset! 

Instead, we use **Imputation** (filling in the holes). We use `.fillna()` (or `.na.fill()`).

Spark is smart enough to match data types. 
* If you say `df.fillna(0)`, Spark will *only* fill the Nulls in integer/double columns with `0`.
* If you say `df.fillna("Unknown")`, Spark will *only* fill the Nulls in string columns with `"Unknown"`.

In [ ]:
print("--- Strategy 3: Filling Nulls with Defaults ---")

# Let's fill missing string columns with 'Unknown' 
# and missing numeric columns (salary) with 0.0

df_filled = df_messy \
    .fillna("Unknown") \
    .fillna(0.0)

df_filled.show()
# Look at Bob's salary (0.0) and Charlie's department (Unknown)!

## 3. Replacing Fake Nulls: `replace()`

Look closely at Eve (Row 5). Her department isn't technically a Spark `null`. It is a string that literally says `"N/A"`. 

If you run `.dropna()` or `.fillna()`, Eve will be completely ignored because Spark thinks `"N/A"` is just a normal, valid word!

Data Engineers must use `.replace()` to convert these "fake" nulls into either real Nulls or standardized values.

In [ ]:
print("--- Strategy 4: Replacing 'Fake' Nulls ---")

# We want to replace the string "N/A" with "Unknown"
# Syntax: replace("old_value", "new_value", subset=["column_name"])

df_replaced = df_filled.replace("N/A", "Unknown", subset=["department"])
df_replaced.show()
# Eve's department is now standardized!

## 4. Null-Safe Operations

What if you don't want to drop or fill the data, but you want to filter based on it? For example, *"Show me all users who DO NOT have a department assigned."*

In Python, you might think to write: `df.filter(F.col("department") == None)`. **This will fail.**

In Spark, `Null` means "Unknown." If you ask Spark, *"Is Unknown equal to Unknown?"*, the mathematical answer is actually *Unknown*. 

To safely check for nulls, you must use **Null-Safe Methods**:
* `F.col("col_name").isNull()`
* `F.col("col_name").isNotNull()`

In [ ]:
print("--- Null-Safe Filtering ---")

# We must use the original df_messy because we haven't filled the nulls in it yet.

print("Users with NO department assigned:")
df_no_dept = df_messy.filter(F.col("department").isNull())
df_no_dept.show()

print("Users WITH a perfectly valid name:")
df_valid_name = df_messy.filter(F.col("name").isNotNull())
df_valid_name.show()

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Handling missing data is a team effort, but the responsibilities differ drastically.

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Prevention** | Sets `NOT NULL` constraints on SQL tables so bad data is completely rejected at the door. | **Often pulls from Data Lakes/APIs where constraints don't exist. Must handle the mess in code.** | Expects the data to be perfectly clean by the time they get it. |
| **Imputation Strategy** | Doesn't usually impute data; just stores what is given. | **Applies business rules:** Fills missing textual categories with "Unknown" or drops fully corrupt rows. | **Applies statistical rules:** Fills missing salaries with the *mathematical mean or median* to prevent biasing the AI model. |
| **Action** | `ALTER TABLE...` | `df.fillna("Unknown")` | `df['salary'].fillna(df['salary'].mean())` |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Relies heavily on `.dropna(how="any")`. They build pipelines that successfully run without crashing, but the business complains that 30% of their daily sales records went missing because the Junior DE aggressively deleted any row with a blank zip code.
2. **Mid-Level DE (Your Current Phase):** Masters `.fillna()` and `.replace()`. They work with business analysts to determine exactly *what* default values should be used (e.g., filling a missing discount code with `0` instead of dropping the sale).
3. **Senior DE:** Implements robust **Data Quality Monitoring**. Rather than just filling nulls silently, they write PySpark logic that counts the percentage of nulls. If a column suddenly goes from 1% Null to 80% Null (meaning a sensor upstream broke), the Senior DE's pipeline halts and alerts the team *before* the bad data poisons the data warehouse.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Data Scientists spend 80% of their time cleaning data. A great Data Engineering team reduces this to 20% by handling Nulls, unifying dates, and casting strings during the core ETL pipeline. This allows Data Scientists to focus on what they do best: building algorithms.

## 🔑 Key Takeaways

- **Nulls Destroy Math:** Any operation involving a `Null` results in a `Null`. They must be handled.
- **`dropna()`:** The aggressive approach. Removes rows containing missing data. Use `subset` to only drop if critical columns are missing.
- **`fillna()`:** The patching approach. Spark will intelligently fill strings with strings and numbers with numbers.
- **`replace()`:** Used to fix string values that *represent* missing data (like "N/A" or "null") but aren't actually Spark native Nulls.
- **Null-Safe Checking:** Never use `== None` or `== Null`. Always use `.isNull()` or `.isNotNull()`.

## 📝 Knowledge Check Quiz

**Question 1:** You run the command `df.dropna()`. By default, what does this command do?
A) It drops the entire DataFrame from the cluster memory.
B) It drops only the columns that have null values.
C) It drops any row that contains at least one Null value in any column (equivalent to `how="any"`).
D) It drops only rows where *every single column* is Null.

**Question 2:** You have a DataFrame with a mix of string columns (names) and integer columns (ages). You run `df.fillna(0)`. What happens?
A) Spark crashes because it tries to put the integer 0 into a string column.
B) Spark intelligently fills the missing integer values (ages) with 0, and ignores the missing string values (names).
C) Spark converts all string columns into integers and fills them with 0.
D) Spark replaces every single Null in the entire DataFrame with the string "0".

**Question 3:** Why will the code `df.filter(F.col("age") == None)` fail to find rows with missing ages?
A) Because you must use the word `Null` instead of `None`.
B) Because Python requires single quotes around None.
C) Because you cannot use mathematical equals signs to evaluate a Null. You must use the null-safe operation `F.col("age").isNull()`.
D) Because filtering on ages is not allowed.

---

*Answers: 1: C, 2: B, 3: C*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You've learned how to create DataFrames, transform them, and clean them. 

Now it is time to actually calculate business insights! How do we find the total revenue for the year, or the average salary by department? 

In the next lesson, **10.15 Aggregations**, we will learn how to use `.groupBy()` and `.agg()` to crush Petabytes of data down into meaningful summary reports. See you there!